In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
def jacobi_preconditioned_conjugate_gradient(A: np.ndarray, b: np.ndarray, x0: np.ndarray = None, tol: float = 1e-10, max_iter: int = 1000) -> np.ndarray:
    """
    Solve the linear system Ax = b using the Jacobi Preconditioned Conjugate Gradient method.
    Parameters:
    A : np.ndarray
        Symmetric positive-definite matrix.
    b : np.ndarray
        Right-hand side vector.
    x0 : np.ndarray, optional
        Initial guess for the solution (default is a zero vector).
    tol : float, optional
        Tolerance for convergence (default is 1e-10).
    max_iter : int, optional
        Maximum number of iterations (default is 1000).
    """

    # number of rows
    n = A.shape[0]

    if x0 is None:
        x = np.zeros(n)
    else:
        x = x0.copy()

    r = b - A @ x
    M_inv = 1 / np.diag(A)  # Jacobi preconditioner (inverse of diagonal elements)
    z = M_inv * r
    p = z.copy()
    rs_old = np.dot(r, z)

    for i in range(max_iter):
        Ap = A @ p
        alpha = rs_old / np.dot(p, Ap)

        x = x + alpha * p
        r = r - alpha * Ap
        
        z = M_inv * r
        rs_new = np.dot(r, z)
        if np.sqrt(rs_new) / np.sqrt(rs_old) < tol:
            print("Converged")
            break

        beta = rs_new / rs_old

        p = z + beta * p

        rs_old = rs_new

    return x   

# Example usage
if __name__ == "__main__":
    # Define a symmetric positive-definite matrix A and vector b
    A = np.array([[4, 1], [1, 3]], dtype=float)
    b = np.array([1, 2], dtype=float)

    # Solve for x in Ax = b
    import time
    start_time = time.time()
    x = jacobi_preconditioned_conjugate_gradient(A, b)
    end_time = time.time()
    print("Time taken:", (end_time - start_time) / 60, "minutes")
    assert np.allclose(x, np.array([0.090909, 0.636364]))
    print("Solution x:", x)

In [ ]:
def check_spd_properties(A: np.ndarray) -> None:
    print(f"Matrix shape: {A.shape}")
    print(f"Matrix:\n{A}\n")

    # Check symmetry
    is_sym = np.allclose(A, A.T, atol=1e-10)
    print(f"✓ Symmetric: {is_sym}")

    # Check SPD via eigenvalues
    eigvals = np.linalg.eigvalsh(A)
    print(f"Eigenvalues: {eigvals}")
    is_spd = np.all(eigvals > 1e-10)
    print(f"✓ Positive definite (all eigenvalues > 0): {is_spd}")

    # Condition number
    if is_spd:
        cond_num = np.max(eigvals) / np.min(eigvals)
        print(f"Condition number: {cond_num:.2e}")

    # Check diagonal dominance
    diag = np.diag(A)
    row_sums = np.sum(np.abs(A), axis=1) - np.abs(diag)
    is_dd = np.all(np.abs(diag) > row_sums)
    print(f"✓ Diagonally dominant: {is_dd}")

In [ ]:
def load_matrix_from_binary_check_spd(file_path: str) -> None:
    """Load a square matrix from a binary file with the specified format.
       the function to check the dense matrix properties: symmetric, SPD, condition number, diagonal dominance.
    """
    with open("../data/matrix.bin", "rb") as f:
        n_rows = np.fromfile(f, dtype=np.int32, count=1)[0]
        n_cols = np.fromfile(f, dtype=np.int32, count=1)[0]
        assert n_rows == n_cols, f"Not square: {n_rows}x{n_cols}"
        A = np.fromfile(f, dtype=np.float64, count=n_rows * n_cols).reshape(n_rows, n_cols)

    check_spd_properties(A)

In [ ]:
DATA_PATH = "../output/"

matrix_bin_path = DATA_PATH + "matrix_dense.bin"
matrix_mtx_path = DATA_PATH + "csr_matrix_full.mtx"

In [ ]:
# check the properties of the matrix loaded from the .bin file
load_matrix_from_binary_check_spd(matrix_bin_path)

In [ ]:
def load_matrix_from_mtx_and_check_spd(file_path: str) -> None:
    """Load a square matrix from a Matrix Market (.mtx) file int a compressed format."""
    from scipy.io import mmread
    A = mmread(file_path).toarray()
    n_rows, n_cols = A.shape
    assert n_rows == n_cols, f"Not square: {n_rows}x{n_cols}"
    
    check_spd_properties(A)

In [ ]:
# check the properties of the matrix loaded from the .mtx file
load_matrix_from_mtx_and_check_spd(matrix_mtx_path)

In [ ]:
def compute_speedups(serial_time: list, parallel_times: list) -> list:
    """
    Compute speedups given serial and parallel execution times.
    Parameters:
        serial_time : Execution time of the serial implementation.
        parallel_times : Execution times of the parallel implementations.
    Returns:
        List of speedup values.
    """
    return [serial_time / t for t in parallel_times]

def compute_efficiencies(speedups: list, num_processors: list) -> list:
    """
    Compute efficiencies given speedups and number of processors.
    Parameters:
        speedups : List of speedup values.
        num_processors : List of number of processors used.
    Returns:
        List of efficiency values.
    """
    return [s / p for s, p in zip(speedups, num_processors)]

In [ ]:
def read_file_to_csv(file_path: str = "../output/jcgtimes.txt") -> pd.DataFrame:
    """Read the jcgtimes.txt file into a pandas DataFrame."""
    times = pd.read_csv(file_path, sep=r'\s+')
    times_final = times.groupby(by=['#grid_size', 'num_processes'], as_index=False).agg({
        'max_time_seconds': 'mean',
        'min_time_seconds': 'mean',
    }).rename(columns={
        'max_time_seconds': 'avg_max_time_seconds',
        'min_time_seconds': 'avg_min_time_seconds'
    })
    return times_final

In [ ]:
def plot_speedup_efficiency(
        file_path: str = "../output/jcgtimes.txt",
        title: str = "Block JCG", with_ideal: bool = True,
        output_file_path: str = None,
        combined: bool = False) -> None:
    """
    Plot speedup and efficiency from timing data.
    Parameters:
        file_path : Path to the timing data file.
        title : Title for the plots.
        with_ideal : Whether to plot ideal speedup and efficiency lines.
        output_file_path : Path to save the output plot (if None, display the plot).
    Returns:
        None
    """
    results_df = read_file_to_csv(file_path)

    # Create figures
    plt.figure(figsize=(12, 6))
    plt.suptitle(f"Speedup and Efficiency for All Problem Sizes - {title}", fontsize=16)

    # Speedup plot
    plt.subplot(1, 2, 1)
    for size in results_df["#grid_size"].unique():
        if size <= 200:
            continue
        subset = results_df[results_df["#grid_size"] == size]
        subset = subset.sort_values("num_processes")
        times = subset["avg_max_time_seconds"].tolist()
        num_processors = subset["num_processes"].tolist()
        
        serial_time = times[0]
        speedups = compute_speedups(serial_time, times)
        plt.plot(num_processors, speedups, marker='*', label=f'Grid {size}')
    # ideal speedup line
    if with_ideal:
        ideal_speedup = list(range(1, max(results_df["num_processes"]) + 1))
        plt.plot(ideal_speedup, ideal_speedup, 'k--', label='Ideal Speedup')
    plt.title('Speedup vs Number of Processors')
    plt.xlabel('Number of Processors')
    plt.ylabel('Speedup')
    plt.xticks(num_processors)
    # plt.yticks(num_processors)
    plt.legend()
    plt.grid(True)
    if not combined:
        plt.tight_layout()
        plt.savefig("../output/speedup_" + output_file_path) if output_file_path else None

    # Efficiency plot
    plt.subplot(1, 2, 2)
    for size in results_df["#grid_size"].unique():
        if size <= 200:
            continue
        subset = results_df[results_df["#grid_size"] == size]
        subset = subset.sort_values("num_processes")
        times = subset["avg_max_time_seconds"].tolist()
        num_processors = subset["num_processes"].tolist()
        
        serial_time = times[0]
        speedups = compute_speedups(serial_time, times)
        efficiencies = compute_efficiencies(speedups, num_processors)
        
        plt.plot(num_processors, efficiencies, marker='*', label=f'Grid {size}')
    # ideal efficiency line
    if with_ideal:
        ideal_efficiency = [1.0 for _ in range(1, max(results_df["num_processes"]) + 1)]
        plt.plot(ideal_speedup, ideal_efficiency, 'k--', label='Ideal Efficiency')
    plt.title('Efficiency vs Number of Processors')
    plt.xlabel('Number of Processors')
    plt.ylabel('Efficiency')
    plt.xticks(num_processors)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    if not combined:
        plt.savefig("../output/efficiency_" + output_file_path) if output_file_path else None
    else:
        plt.savefig("../output/speedup_efficiency_" + output_file_path) if output_file_path else None
    plt.show()

In [ ]:
plot_speedup_efficiency(
    "../output/jcgtimes_block.txt", title="Block JCG", with_ideal=True,
    output_file_path="block_jcg.eps",
    combined=True
)

In [ ]:
plot_speedup_efficiency(
    "../output/jcgtimes_block.txt", title="Block JCG", with_ideal=False,
    output_file_path="block_jcg_no_ideal.eps",
    combined=True
)

In [ ]:
plot_speedup_efficiency(
    "../output/jcgtimes_nblock.txt", title="Non Block JCG", with_ideal=True,
    output_file_path="nonblock_jcg.eps",
    combined=True
)

In [ ]:
plot_speedup_efficiency(
    "../output/jcgtimes_nblock.txt", title="Non Block JCG", with_ideal=False,
    output_file_path="nonblock_jcg_no_ideal.eps",
    combined=True
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

def plot_convergence_analysis(baseline_file: str = "../output/residual_baseline.txt",
                              pipelined_file: str = "../output/residual_pipelined.txt",
                              output_file_path: str = "../plots/convergence_analysis.eps") -> None:
    """
    Plot convergence analysis for the research report.
    Assumes files have columns: #iteration  residual_norm
    """
    
    # 1. Load Data (skiprows/comment handles the '#' in your C output)
    try:
        baseline_data = pd.read_csv(baseline_file, sep=r'\s+', comment='#', names=['iteration', 'residual'])
        pipelined_data = pd.read_csv(pipelined_file, sep=r'\s+', comment='#', names=['iteration', 'residual'])
    except FileNotFoundError as e:
        print(f"Error: Could not find result files. {e}")
        return

    # 2. Setup Plot Style
    plt.style.use('seaborn-v0_8-paper') # Or 'ggplot'
    plt.figure(figsize=(8, 5))

    # 3. Plot trajectories
    # We use markevery=10 so markers don't overlap too much
    plt.semilogy(baseline_data['iteration'], baseline_data['residual'], 
                 label='Baseline Jacobi-CG', 
                 color='blue', linestyle='-', marker='o', 
                 markevery=10, markersize=5, linewidth=1.5)

    plt.semilogy(pipelined_data['iteration'], pipelined_data['residual'], 
                 label='Pipelined Jacobi-CG', 
                 color='red', linestyle='--', marker='s', 
                 markevery=10, markersize=5, linewidth=1.5)

    # 4. Add Reference Tolerance Line (10^-10)
    plt.axhline(y=1e-10, color='gray', linestyle=':', alpha=0.7, label='Target Tolerance ($10^{-10}$)')

    # 5. Labels and Titles (Standard for academic papers)
    plt.title('Residual Convergence: Baseline vs. Pipelined JCG', fontsize=14)
    plt.xlabel('Iteration Number ($k$)', fontsize=12)
    plt.ylabel(r'Relative Residual $\frac{||r_k||_2}{||r_0||_2}$', fontsize=12)
    
    plt.legend(loc='upper right', frameon=True)
    plt.grid(True, which="both", ls="-", alpha=0.2)
    
    # 6. Save and clean up
    # Ensure plots directory exists
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    
    plt.tight_layout()
    plt.savefig(output_file_path, format='eps', dpi=300)
    print(f"Plot saved to: {output_file_path}")
    plt.show()

In [ ]:
plot_convergence_analysis()